# 06 - QLoRA tune: `qwen2.5-3b-csai415`  (D4 Task 3b - Owner: WAFIQ)

**Goal:** QLoRA-tune `Qwen2.5-3B-Instruct` on the curated arXiv Q/A set so the answerer cites better/faithfully,
then **merge -> GGUF 4-bit -> serve via Ollama** as `qwen2.5-3b-csai415`. Because `answer.py` selects the model
by the `CSAI415_ANSWERER` env var, the tuned model is a **config swap, not a new interface** - the executor and
eval harness stay byte-identical to zero-shot. See `briefs/D3_D4_TASKS.md` (Task 3, Phase B).

**The one rule that makes this work:** the model is trained on the **exact** chat format `src/csai415/answer.py`
emits at inference (system prompt + numbered-source user template, assistant = `[n]`-cited answer). Train/infer
format drift is the #1 way a small-model tune silently regresses, so the format constants below are copied
verbatim from `answer.py` and Cell 4 asserts they still match if the repo is importable.

**Approach = compare, then defend the winner (rubric).** We sweep the LoRA rank - the canonical QLoRA capacity
knob - at fixed epochs, pick the winner on held-out eval loss, and keep the per-epoch eval-loss curve so the
overfit story on a small (~125-row) set is evidenced, not asserted. The winning adapter is what we merge + serve.

**Runtime:** Colab **T4** (16 GB). Turing has **no bf16** (needs Ampere 8.0+), so everything below runs in
**fp16** - `bf16=True` would hard-error on a T4. Epochs are set to **2** (not 3): ~112 train rows learn the
citation format in 1-2 passes; a 3rd epoch mostly overfits a set this small. The per-epoch eval loss (logged
by the sweep) tells you if 2 was enough - bump back up only if eval loss is still falling steeply at epoch 2.

**Inputs:** `data/train/qa_train.jsonl` (125 rows, disjoint from gold - leakage-checked). **Outputs:** merged
fp16 -> `qwen2.5-3b-csai415.Q4_K_M.gguf` + Ollama model + numbers for `reports/D4/tuning_card.md`.

> Downstream: Abdulrahman (T7) runs **base-vs-tuned** through `evaluate_answers` by flipping `CSAI415_ANSWERER`
> between `qwen2.5:3b-instruct` and `qwen2.5-3b-csai415` - that produces the final delta table. Nothing here
> touches his files.

## 1 - Environment (Colab)
Pin the QLoRA stack. Skip if your runtime already has it.

In [ ]:
%pip install -q -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "bitsandbytes>=0.43" \
    "accelerate>=0.33" "datasets>=2.20" "pandas"
import torch, transformers, trl, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("bf16 supported:", torch.cuda.is_bf16_supported(), "(False on T4 -> we use fp16)")
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)

## 2 - Config - all knobs in one place
`SWEEP_GRID` is the comparison: three LoRA ranks (8 / 16 / 32) at fixed **2 epochs** and lr. Everything else is
held constant so rank is the only moving part. The winner (Cell 7) feeds merge + serve; these values + the
winner go into the tuning card. If the T4 is too slow, drop rank 32 (most overfit-prone on 112 rows) for a
2-point sweep.

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"   # base; Apache-2.0 (record license in the card)
OLLAMA_NAME  = "qwen2.5-3b-csai415"         # must match CSAI415_ANSWERER for the tuned row
TRAIN_PATH   = "data/train/qa_train.jsonl"  # upload to Colab or mount Drive (Cell 3)
OUT_ROOT     = "outputs"                     # per-rank adapters land in OUT_ROOT/qlora-r{rank}
MERGED_DIR   = "outputs/merged-qwen2.5-3b-csai415"
GGUF_OUT     = "outputs/qwen2.5-3b-csai415.Q4_K_M.gguf"
GGUF_QUANT   = "Q4_K_M"   # 4-bit serving target (matches the 'GGUF 4-bit' decision)
SEED         = 42

# --- the sweep: (lora_rank, lora_alpha) at fixed epochs; alpha = 2*rank convention ---
SWEEP_GRID   = [(8, 16), (16, 32), (32, 64)]
EPOCHS       = 2      # T4-friendly: 112 rows learn the format in 1-2 passes; 3rd epoch mostly overfits
LR           = 2e-4

# held constant across the sweep
MAX_SEQ_LEN  = 2048   # system + up to 3 ctx (<=1200 chars each) + answer fits comfortably
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
WARMUP_RATIO = 0.03
PER_DEV_BS   = 1
GRAD_ACCUM   = 8      # effective batch = 8

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3 - Get the training data into the runtime
Pick ONE: upload `qa_train.jsonl`, mount Drive, or clone the repo. The file must land at `TRAIN_PATH`.

In [ ]:
import os
if not os.path.exists(TRAIN_PATH):
    # Option A - manual upload (Colab):
    #   from google.colab import files; up = files.upload()  # choose qa_train.jsonl
    #   os.makedirs('data/train', exist_ok=True)
    #   os.rename('qa_train.jsonl', TRAIN_PATH)
    # Option B - clone the repo (private -> use a token):
    #   !git clone https://github.com/waf-iq/special-topics.git && cd special-topics
    raise FileNotFoundError(f"Put qa_train.jsonl at {TRAIN_PATH} (see options above).")
print("found:", TRAIN_PATH, os.path.getsize(TRAIN_PATH), "bytes")

## 4 - Format = `answer.py` (verbatim) - the contract
`SYSTEM_PROMPT`, the user template and `MAX_CTX_CHARS` are **copied from `src/csai415/answer.py`**. The optional
assert imports the real module (if the repo is on `sys.path`) and fails loudly on any drift, so this notebook
can never train on a format the deployed answerer doesn't use.

In [ ]:
# --- copied verbatim from src/csai415/answer.py (keep in sync) ---
REFUSAL = "I could not find relevant information in the retrieved context."
MAX_CTX_CHARS = 1200
SYSTEM_PROMPT = (
    "You are a precise research assistant. Answer the question using ONLY the numbered "
    "sources provided. After each claim, cite the source you used by writing its number in "
    "square brackets, for example [1] or [2]. Always use the actual digit of the source - "
    "never write the literal letter n or 'n.1'. Keep the answer concise. If the sources do "
    f'not contain the answer, reply with exactly "{REFUSAL}" and nothing else. Do not use '
    "any outside knowledge."
)

def build_user(question, contexts):
    """Mirror answer.py:_numbered_source - the exact user turn the model sees at inference."""
    sources = "\n".join(f"[{i}] {(c or '')[:MAX_CTX_CHARS]}" for i, c in enumerate(contexts, 1))
    return (
        f"Sources:\n{sources}\n\nQuestion: {question}\n\n"
        "Answer (cite each claim with its source's number in square brackets, e.g. [1]):"
    )

# --- optional drift guard: assert against the real module if importable ---
try:
    import sys; sys.path.insert(0, 'src')
    from csai415 import answer as _a
    assert SYSTEM_PROMPT == _a._SYSTEM_PROMPT, 'SYSTEM_PROMPT drifted from answer.py'
    assert MAX_CTX_CHARS == _a._MAX_CTX_CHARS, 'MAX_CTX_CHARS drifted from answer.py'
    print('OK - format constants match src/csai415/answer.py')
except Exception as e:
    print('(skipped drift assert - repo not importable here):', e)

## 5 - Build the chat dataset
Each training row -> `[system, user, assistant]`. Qwen's own chat template renders the turns; we train **only on
the assistant completion** (the collator in Cell 6 masks the prompt), so the model learns to *produce* cited
answers, not to echo the sources. One fixed 90/10 split (seed 42) is reused for every sweep point.

In [ ]:
import json
from datasets import Dataset

rows = [json.loads(l) for l in open(TRAIN_PATH, encoding='utf-8') if l.strip()]
print(f"{len(rows)} training rows")

def to_messages(r):
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user(r["question"], r["contexts"])},
        {"role": "assistant", "content": r["answer"]},
    ]}

ds = Dataset.from_list([to_messages(r) for r in rows]).shuffle(seed=SEED)
split = ds.train_test_split(test_size=0.1, seed=SEED)  # held-out slice to pick the sweep winner
train_ds, eval_ds = split["train"], split["test"]
print("train", len(train_ds), "| eval", len(eval_ds))

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

sample = tok.apply_chat_template(train_ds[0]["messages"], tokenize=False)
print("\n--- one rendered example ---\n", sample[:1200])
lens = [len(tok.apply_chat_template(x["messages"], tokenize=True)) for x in train_ds]
print(f"\ntoken lengths: min {min(lens)} | max {max(lens)} | MAX_SEQ_LEN {MAX_SEQ_LEN}")
assert max(lens) <= MAX_SEQ_LEN, 'some examples exceed MAX_SEQ_LEN - raise it or trim contexts'

## 6 - `train_one(rank, alpha)` - one sweep point
Loads the base fresh in 4-bit (NF4 + double-quant, **fp16 compute** for T4) each call so adapters never
contaminate each other, attaches LoRA, trains with completion-only loss, and returns the adapter dir +
final/per-epoch eval loss. Frees the model before returning so three runs fit on one GPU.

In [ ]:
import gc, time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"   # Qwen2.5 assistant-turn marker
collator = DataCollatorForCompletionOnlyLM(response_template=RESPONSE_TEMPLATE, tokenizer=tok)

def train_one(rank, alpha):
    out_dir = f"{OUT_ROOT}/qlora-r{rank}"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                                 device_map="auto", torch_dtype=torch.float16)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
                      target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora)
    cfg = SFTConfig(
        output_dir=out_dir, num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=PER_DEV_BS, gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine", optim="paged_adamw_8bit",
        logging_steps=5, eval_strategy="epoch", save_strategy="no",
        fp16=True, max_seq_length=MAX_SEQ_LEN, packing=False, seed=SEED, report_to="none",
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds,
                         processing_class=tok, data_collator=collator)
    t0 = time.time(); trainer.train(); secs = time.time() - t0
    trainer.save_model(out_dir); tok.save_pretrained(out_dir)
    epoch_evals = [round(h["eval_loss"], 4) for h in trainer.state.log_history if "eval_loss" in h]
    final_eval = epoch_evals[-1] if epoch_evals else float("nan")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return {"rank": rank, "alpha": alpha, "out_dir": out_dir,
            "final_eval_loss": final_eval, "epoch_eval_losses": epoch_evals, "train_secs": round(secs)}

## 7 - Run the sweep + pick the winner
Three runs (~5-8 min each on a T4). Winner = lowest held-out eval loss; the per-epoch column exposes overfitting
(eval loss turning back up) and tells you whether 2 epochs was enough. Copy this table into the tuning card -
it's the 'compared approaches' evidence.

In [ ]:
import pandas as pd
results = [train_one(r, a) for (r, a) in SWEEP_GRID]
df = pd.DataFrame(results)[["rank","alpha","final_eval_loss","epoch_eval_losses","train_secs","out_dir"]]
df = df.sort_values("final_eval_loss").reset_index(drop=True)
print(df.to_string(index=False))
print("\n=== markdown for the tuning card ===\n")
print(df[["rank","alpha","final_eval_loss","epoch_eval_losses","train_secs"]].to_markdown(index=False))

BEST = df.iloc[0]
BEST_DIR, BEST_RANK = BEST["out_dir"], int(BEST["rank"])
print(f"\nWINNER -> rank {BEST_RANK}  (eval_loss {BEST['final_eval_loss']})  @ {BEST_DIR}")

## 8 - Smoke test the winning adapter
Reload the winner onto a fresh 4-bit base and generate through the **same** chat format. Must emit in-range `[n]`
markers and refuse on out-of-context questions - the two behaviours the tune must not break.

In [ ]:
import re
from peft import PeftModel

_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=_bnb,
                                             device_map="auto", torch_dtype=torch.float16)
smoke_model = PeftModel.from_pretrained(_base, BEST_DIR)

def ask_tuned(question, contexts, max_new_tokens=320):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":build_user(question, contexts)}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(smoke_model.device)
    out = smoke_model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0,
                               pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

raw = rows[0]
print("GOLD :", raw["answer"][:400], "\n")
ans = ask_tuned(raw["question"], raw["contexts"])
print("TUNED:", ans)
print("cited:", [int(d) for grp in re.findall(r'\[([^\[\]]*)\]', ans) for d in re.findall(r'\d+', grp)])
print("\nrefusal check:", ask_tuned("What is the capital of France?", raw["contexts"]))
del smoke_model, _base; gc.collect(); torch.cuda.empty_cache()

## 9 - Merge winner -> fp16
Reload the base in fp16 (no 4-bit), merge the **winning** adapter, save standalone weights for GGUF conversion.

In [ ]:
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, BEST_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True); tok.save_pretrained(MERGED_DIR)
print(f"merged fp16 (rank {BEST_RANK}) ->", MERGED_DIR)
del merged, base; gc.collect(); torch.cuda.empty_cache()

## 10 - Convert -> GGUF (4-bit `Q4_K_M`)
`llama.cpp` converts the merged HF model to fp16 GGUF, then quantizes to `Q4_K_M` - the serving target Ollama loads.

In [ ]:
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
    !pip -q install -r llama.cpp/requirements.txt
    !cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF >/dev/null 2>&1 && \
     cmake --build llama.cpp/build --target llama-quantize -j >/dev/null 2>&1

F16 = 'outputs/qwen2.5-3b-csai415.f16.gguf'
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16} --outtype f16
!./llama.cpp/build/bin/llama-quantize {F16} {GGUF_OUT} {GGUF_QUANT}
print('GGUF ->', GGUF_OUT)
!ls -lh {GGUF_OUT}

## 11 - Serve via Ollama as `qwen2.5-3b-csai415`
Run these **where Ollama lives** (your laptop / the demo box) after downloading the GGUF from Colab. The Modelfile
name **must** equal `CSAI415_ANSWERER=qwen2.5-3b-csai415` so the eval harness picks up the tuned model unchanged.

In [ ]:
modelfile = f'''FROM ./{os.path.basename(GGUF_OUT)}
PARAMETER temperature 0.0
PARAMETER num_predict 320
PARAMETER stop "<|im_end|>"
TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
"""
SYSTEM """{SYSTEM_PROMPT}"""
'''
open('Modelfile','w',encoding='utf-8').write(modelfile)
print(modelfile)
# Then, in a shell next to the GGUF + Modelfile:
#   ollama create qwen2.5-3b-csai415 -f Modelfile
#   CSAI415_ANSWERER=qwen2.5-3b-csai415 ollama run qwen2.5-3b-csai415 'smoke test'

## 12 - Hand-off

1. **Download** `qwen2.5-3b-csai415.Q4_K_M.gguf` from Colab; `ollama create` it on the demo box (Cell 11).
2. **Fill `reports/D4/tuning_card.md`**: paste the Cell 7 sweep table (the 'compared approaches'), the winning
   rank, plus GPU (T4) + wall-clock, final losses, and base-model license (Qwen2.5 = Apache-2.0).
3. **Hand to Abdulrahman (T7)** for the base-vs-tuned delta table - he runs `evaluate_answers` twice:
   `CSAI415_ANSWERER=qwen2.5:3b-instruct` then `=qwen2.5-3b-csai415`. No code change; pure config swap.
4. Sanity before declaring done: tuned model still **refuses** out-of-context questions and emits **in-range**
   `[n]` (don't let the tune trade faithfulness for fluency - that's the headline risk of a small-set SFT).

> Open question for the card's discussion: does 4-bit GGUF quantization erode faithfulness vs the merged fp16
> model? If T7's harness shows a gap, note it and consider `Q5_K_M`.